# 01 · Features Geoespaciales + Modelo Baseline — Savia Salud EPS
**Fase 1:** Ingeniería de features geoespaciales y entrenamiento modelo baseline  
**Input:** `Data/processed/YYYYMMDD_geo_eda_fase0.parquet` (output Fase 0)  
**Output:** `Data/processed/YYYYMMDD_geo_features.parquet` + `models/artifacts/YYYYMMDD_geo_xgboost_v1.json`  
**Reglas activas:** Solo SELECT · Sin PII · División temporal · Anti-leakage · RANDOM_STATE=42

## Índice
1. Cargar datos de Fase 0
2. División temporal train / val / test
3. Calcular tasas históricas SOBRE TRAIN (anti-leakage)
4. Construir features geoespaciales
5. Análisis de features: correlación con target
6. Entrenamiento modelo XGBoost baseline
7. Evaluación de métricas
8. Importancia de features (SHAP)
9. Guardar artefactos

## 0 · Configuración

In [ ]:
import logging
import sys
import warnings
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

BASE_DIR  = Path("d:/Users/jcardonr/Documents/Savia")
PROC_DIR  = BASE_DIR / "Data" / "processed"
FIG_DIR   = BASE_DIR / "reports" / "figures"
MODEL_DIR = BASE_DIR / "models" / "artifacts"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BASE_DIR))

from src.data.geo_features import (
    build_geo_features,
    compute_glosa_rate_by_municipio,
    compute_hhi_dx_by_municipio,
    get_geo_feature_names,
)

TODAY        = date.today().strftime("%Y%m%d")
RANDOM_STATE = 42
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 5)})
logger.info("Entorno listo · %s", TODAY)

## 1 · Cargar datos de Fase 0

In [ ]:
# Buscar el parquet más reciente de Fase 0
fase0_files = sorted(PROC_DIR.glob("*_geo_eda_fase0.parquet"))
if not fase0_files:
    raise FileNotFoundError("No se encontró archivo de Fase 0. Ejecuta primero 00_eda_geoespacial.ipynb")

fase0_path = fase0_files[-1]  # más reciente
logger.info("Cargando: %s", fase0_path)
df_base = pd.read_parquet(fase0_path)
df_base["fecha_radicacion"] = pd.to_datetime(df_base["fecha_radicacion"])

logger.info("Dataset Fase 0: %d filas · %d cols", len(df_base), len(df_base.columns))
logger.info("Período: %s → %s", df_base["fecha_radicacion"].min().date(), df_base["fecha_radicacion"].max().date())

# Cargar catálogo de sedes IPS
sedes_files = sorted((BASE_DIR / "Data" / "raw").glob("*_cnt_prestador_sedes.parquet"))
if not sedes_files:
    raise FileNotFoundError("No se encontró catálogo de sedes. Ejecuta primero 00_eda_geoespacial.ipynb")
df_sedes = pd.read_parquet(sedes_files[-1])
logger.info("Sedes IPS: %d registros", len(df_sedes))

print(f"\nDistribución de target:")
print(df_base["target"].value_counts().rename({0: "Auditada", 1: "Glosada", 2: "Devuelta"}))

## 2 · División temporal train / val / test

**Regla:** NUNCA split aleatorio. División por `fecha_radicacion`:  
- Train: primeros 60% del período  
- Val:   siguiente 20%  
- Test:  último 20%  

In [ ]:
df_sorted = df_base.sort_values("fecha_radicacion").reset_index(drop=True)

n = len(df_sorted)
n_train = int(n * 0.60)
n_val   = int(n * 0.20)

df_train = df_sorted.iloc[:n_train].copy()
df_val   = df_sorted.iloc[n_train : n_train + n_val].copy()
df_test  = df_sorted.iloc[n_train + n_val :].copy()

for split_name, split_df in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    logger.info(
        "%s: %d filas · %s → %s · Glosa: %.1f%%",
        split_name, len(split_df),
        split_df["fecha_radicacion"].min().date(),
        split_df["fecha_radicacion"].max().date(),
        (split_df["target"] == 1).mean() * 100,
    )

## 3 · Calcular tasas históricas SOBRE TRAIN (anti-leakage)

In [ ]:
# ─── Hallazgos EDA que ajustan la Fase 1 ─────────────────────────────────────
# - GPS propio afiliado: solo 2.2% → distancia usa centroide municipal (fallback dominante)
# - GPS IPS: solo 6.1% → distancia también usará centroide del municipio de la IPS
# - nivel1_en_municipio: varianza CERO (todos los municipios tienen IPS nivel 1) → excluir
# - glosa_rate_municipio: 10 municipios > 13% → sí discriminativa
# ─────────────────────────────────────────────────────────────────────────────

# Calcular SOLO sobre train (anti-leakage)
glosa_rates  = compute_glosa_rate_by_municipio(df_train)
hhi_dict     = compute_hhi_dx_by_municipio(df_train)
global_glosa = (df_train["target"] == 1).mean()
global_hhi   = df_train.groupby(
    df_train["municipio_residencia"].str.strip().str.upper()
)["codigo_dx"].apply(lambda s: float((s.value_counts(normalize=True) ** 2).sum())).mean()

logger.info("Tasa global de glosa en train: %.2f%%", global_glosa * 100)
logger.info("Municipios con tasa histórica: %d", len(glosa_rates))
logger.info("Municipios con HHI diagnóstico: %d", len(hhi_dict))

## 4 · Construir features geoespaciales

In [ ]:
build_kwargs = dict(
    df_sedes=df_sedes,
    glosa_rates=glosa_rates,
    hhi_dict=hhi_dict,
    global_glosa_rate=global_glosa,
    global_hhi=global_hhi,
)

df_train_geo = build_geo_features(df_train, **build_kwargs)
df_val_geo   = build_geo_features(df_val,   **build_kwargs)
df_test_geo  = build_geo_features(df_test,  **build_kwargs)

# Excluir nivel1_en_municipio: varianza cero (todos los municipios tienen IPS nivel 1)
# → no aporta información al modelo, solo ruido
geo_feats = [f for f in get_geo_feature_names() if f != "nivel1_en_municipio"]
logger.info("Features geo activas (%d): %s", len(geo_feats), geo_feats)

# Guardar dataset enriquecido completo
geo_path = PROC_DIR / f"{TODAY}_geo_features.parquet"
pd.concat([df_train_geo, df_val_geo, df_test_geo]).to_parquet(geo_path, index=False)
logger.info("Dataset con features geo guardado: %s", geo_path)

## 5 · Análisis de features: correlación con target

In [ ]:
# Correlación de features geo con target (sobre train)
feats_disponibles = [f for f in geo_feats if f in df_train_geo.columns]
corr = df_train_geo[feats_disponibles + ["target"]].corr()["target"].drop("target").sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#e74c3c" if v > 0 else "#3498db" for v in corr.values]
ax.barh(corr.index, corr.values, color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Correlación de Pearson con target")
ax.set_title("Correlación de features geoespaciales con target (train)")
plt.tight_layout()
plt.savefig(FIG_DIR / "01_correlacion_geo_features.png", bbox_inches="tight")
plt.show()
print(corr.round(4))

In [ ]:
# Distribución de cada feature geo por clase
feats_num = [
    f for f in feats_disponibles
    if f in df_train_geo.columns and pd.api.types.is_numeric_dtype(df_train_geo[f])
]
n_feats = len(feats_num)
n_cols  = 3
n_rows  = max(1, (n_feats + n_cols - 1) // n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows), squeeze=False)
axes = axes.flatten()

for i, feat in enumerate(feats_num):
    plotted = False
    for cls, label, color in [(0, "Auditada", "#3498db"), (1, "Glosada", "#e74c3c"), (2, "Devuelta", "#f39c12")]:
        data = df_train_geo.loc[df_train_geo["target"] == cls, feat].dropna()
        # KDE requiere varianza > 0 y al menos 2 valores distintos
        if len(data) > 1 and data.nunique() > 1:
            try:
                data.plot(kind="kde", ax=axes[i], label=label, color=color)
                plotted = True
            except Exception:
                # Si KDE falla usar histograma normalizado como fallback
                axes[i].hist(data, bins=20, density=True, alpha=0.4, label=label, color=color)
                plotted = True
    if not plotted:
        # Feature constante: mostrar nota
        axes[i].text(0.5, 0.5, "Varianza cero\n(feature constante)",
                     ha="center", va="center", transform=axes[i].transAxes,
                     fontsize=9, color="gray")
    axes[i].set_title(feat, fontsize=9)
    axes[i].legend(fontsize=7)
    axes[i].set_xlabel("")

for j in range(n_feats, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distribución de features geoespaciales por clase", y=1.01)
plt.tight_layout()
plt.savefig(FIG_DIR / "01_dist_geo_features_por_clase.png", bbox_inches="tight")
plt.show()

## 6 · Entrenamiento modelo XGBoost baseline

In [ ]:
# Diagnóstico: verificar que 'target' existe en df_train_geo
print("Columnas df_train_geo:", list(df_train_geo.columns))
print("'target' en df_train_geo:", "target" in df_train_geo.columns)
print("Shape:", df_train_geo.shape)

In [ ]:
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

# Features: geo (sin nivel1_en_municipio) + socioeconómicas (sin PII)
feats_socio           = ["nivel_sisben", "edad"]
feats_categoricas_raw = ["zona_afiliado", "regimen", "grupo_poblacional", "genero"]

all_feats = list(geo_feats) + [f for f in feats_socio if f in df_train_geo.columns]

# Encoding de categóricas
le_dict = {}
for col in feats_categoricas_raw:
    if col in df_train_geo.columns:
        le = LabelEncoder()
        all_vals = pd.concat([df_train_geo[col], df_val_geo[col], df_test_geo[col]]).fillna("DESCONOCIDO")
        le.fit(all_vals)
        for split in [df_train_geo, df_val_geo, df_test_geo]:
            split[col + "_enc"] = le.transform(split[col].fillna("DESCONOCIDO"))
        le_dict[col] = le
        all_feats.append(col + "_enc")

# Con 11M filas, submuestrear train a 1.5M estratificado para velocidad
SAMPLE_SIZE = 1_500_000
if len(df_train_geo) > SAMPLE_SIZE:
    frac = SAMPLE_SIZE / len(df_train_geo)
    df_train_sample = (
        df_train_geo
        .groupby("target", group_keys=False)
        .apply(lambda x: x.sample(frac=frac, random_state=RANDOM_STATE))
        .reset_index(drop=True)   # evita que 'target' quede como índice
    )
    logger.info("Train submuestreado: %d → %d filas (estratificado)", len(df_train_geo), len(df_train_sample))
else:
    df_train_sample = df_train_geo.reset_index(drop=True)

X_train = df_train_sample[all_feats].apply(pd.to_numeric, errors="coerce")
y_train = df_train_sample["target"]
X_val   = df_val_geo[all_feats].apply(pd.to_numeric, errors="coerce")
y_val   = df_val_geo["target"]
X_test  = df_test_geo[all_feats].apply(pd.to_numeric, errors="coerce")
y_test  = df_test_geo["target"]

logger.info("Features totales: %d → %s", len(all_feats), all_feats)
logger.info("Shapes — train: %s · val: %s · test: %s", X_train.shape, X_val.shape, X_test.shape)

# scale_pos_weight sobre train completo para no sesgar
n_neg    = (df_train_geo["target"] != 1).sum()
n_pos    = (df_train_geo["target"] == 1).sum()
scale_pw = n_neg / n_pos
logger.info("scale_pos_weight para Glosada: %.2f  (n_neg=%d / n_pos=%d)", scale_pw, n_neg, n_pos)

# Entrenamiento XGBoost multiclase
model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=10,
    random_state=RANDOM_STATE,
    eval_metric="mlogloss",
    early_stopping_rounds=30,
    n_jobs=-1,
    verbosity=0,
    tree_method="hist",
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50,
)

logger.info("Modelo entrenado · Mejor iteración: %d", model.best_iteration)

## 7 · Evaluación de métricas

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
import matplotlib.ticker as mticker

def evaluar_modelo(model, X, y, nombre: str) -> dict:
    """Evalúa el modelo y retorna métricas estándar del proyecto.

    Args:
        model: Modelo XGBoost entrenado.
        X: Features de evaluación.
        y: Target real.
        nombre: Nombre del split (Train/Val/Test).

    Returns:
        Diccionario con métricas clave.
    """
    y_pred  = model.predict(X)
    y_proba = model.predict_proba(X)

    f1   = f1_score(y, y_pred, average="macro", zero_division=0)
    auc  = roc_auc_score(y, y_proba, multi_class="ovr", average="macro")
    recall_glosada = (y_pred[y == 1] == 1).mean() if (y == 1).any() else 0.0

    print(f"\n{'='*60}")
    print(f"MÉTRICAS — {nombre}")
    print(f"{'='*60}")
    print(f"  F1-macro:          {f1:.4f}")
    print(f"  AUC-ROC (OvR):     {auc:.4f}")
    print(f"  Recall Glosada:    {recall_glosada:.4f}  ← métrica primaria")
    print()
    print(classification_report(y, y_pred, target_names=["Auditada", "Glosada", "Devuelta"], zero_division=0))

    return {"split": nombre, "f1_macro": f1, "auc_roc": auc, "recall_glosada": recall_glosada}


metricas_val  = evaluar_modelo(model, X_val,  y_val,  "Validación")
metricas_test = evaluar_modelo(model, X_test, y_test, "Test")

In [ ]:
# Matriz de confusión normalizada (test)
y_pred_test = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_test, normalize="true")
clases = ["Auditada (0)", "Glosada (1)", "Devuelta (2)"]

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=clases, yticklabels=clases, ax=ax)
ax.set_ylabel("Real")
ax.set_xlabel("Predicho")
ax.set_title("Matriz de Confusión Normalizada — Test")
plt.tight_layout()
plt.savefig(FIG_DIR / "01_confusion_matrix_test.png", bbox_inches="tight")
plt.show()

## 8 · Importancia de features (SHAP)

In [ ]:
try:
    import shap

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_val.fillna(0))

    # SHAP mean |value| por feature (todas las clases)
    if isinstance(shap_values, list):
        importancia_shap = pd.DataFrame(
            np.mean([np.abs(sv) for sv in shap_values], axis=0),
            columns=all_feats,
        ).mean().sort_values(ascending=False)
    else:
        importancia_shap = pd.Series(
            np.abs(shap_values).mean(axis=0), index=all_feats
        ).sort_values(ascending=False)

    top_feats = importancia_shap.head(15)
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.barh(top_feats.index[::-1], top_feats.values[::-1], color="#3498db")
    ax.set_xlabel("Importancia SHAP media |valor|")
    ax.set_title("Top 15 features por importancia SHAP (validación)")

    # Resaltar features geoespaciales
    for patch, feat in zip(ax.patches, top_feats.index[::-1]):
        if feat in geo_feats:
            patch.set_facecolor("#e74c3c")

    plt.tight_layout()
    plt.savefig(FIG_DIR / "01_shap_geo_features.png", bbox_inches="tight")
    plt.show()

    print("\nTop 15 features por importancia SHAP:")
    print(top_feats.round(4).to_string())
    print("\n(Rojo = feature geoespacial)")

except ImportError:
    logger.warning("shap no instalado — usar: pip install shap")
    # Importancia nativa de XGBoost como alternativa
    importancia = pd.Series(model.feature_importances_, index=all_feats).sort_values(ascending=False)
    print("Importancia nativa XGBoost (gain):")
    print(importancia.head(15).round(4))

## 9 · Guardar artefactos

In [ ]:
import json

# Guardar modelo XGBoost en formato nativo .json
model_path = MODEL_DIR / f"{TODAY}_geo_xgboost_v1.json"
model.save_model(str(model_path))
logger.info("Modelo guardado: %s", model_path)

# Guardar metadata del experimento
metadata = {
    "fecha": TODAY,
    "modelo": "XGBoost geo_v1",
    "features": all_feats,
    "features_geo": geo_feats,
    "n_train": len(X_train),
    "n_val": len(X_val),
    "n_test": len(X_test),
    "metricas_val": metricas_val,
    "metricas_test": metricas_test,
    "global_glosa_rate": float(global_glosa),
    "random_state": RANDOM_STATE,
}

meta_path = MODEL_DIR / f"{TODAY}_geo_xgboost_v1_metadata.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
logger.info("Metadata guardada: %s", meta_path)

# Guardar tasas históricas (para aplicar en producción sin recalcular)
rates_path = PROC_DIR / f"{TODAY}_glosa_rates_municipio.json"
with open(rates_path, "w", encoding="utf-8") as f:
    json.dump(glosa_rates, f, indent=2, ensure_ascii=False)
logger.info("Tasas de glosa por municipio guardadas: %s", rates_path)

print("\n" + "="*60)
print("ARTEFACTOS GUARDADOS")
print("="*60)
print(f"  Modelo:    {model_path}")
print(f"  Metadata:  {meta_path}")
print(f"  Tasas geo: {rates_path}")
print(f"  Dataset:   {geo_path}")
print()
print("RESUMEN DE MÉTRICAS (Test):")
print(f"  F1-macro:          {metricas_test['f1_macro']:.4f}")
print(f"  AUC-ROC:           {metricas_test['auc_roc']:.4f}")
print(f"  Recall Glosada:    {metricas_test['recall_glosada']:.4f}")
print()
print(f"Baseline objetivo AUC-ROC ≥ 0.75: {'✓ CUMPLIDO' if metricas_test['auc_roc'] >= 0.75 else '✗ PENDIENTE'}")